# 06 Anomaly Detection - Sample Or Full Dataset

Sample mode defaults to SBERT because BGE is intentionally disabled for quick local runs. Full mode defaults to BGE after the server encode finishes.


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import pandas as pd

sys.path.append('..')
from src.full_pipeline import (
    PROCESSED_PATH,
    EMBEDDING_DIR,
    EXPERIMENTS_DIR,
    parquet_row_count,
    sample_processed_path,
    sample_embedding_dir,
    run_full_anomaly_for_embedding,
)


## 1. Anomaly Configuration


In [ ]:
RUN_MODE = "sample"  # "sample" for local, "full" for server
ROWS_PER_CATEGORY = 1_000

ACTIVE_PROCESSED_PATH = sample_processed_path(ROWS_PER_CATEGORY) if RUN_MODE == "sample" else PROCESSED_PATH
ACTIVE_EMBEDDING_DIR = sample_embedding_dir(ROWS_PER_CATEGORY) if RUN_MODE == "sample" else EMBEDDING_DIR
ACTIVE_EXPERIMENTS_DIR = EXPERIMENTS_DIR / 'local_sample' if RUN_MODE == "sample" else EXPERIMENTS_DIR

ENCODER_TO_TEST = 'bge'
CONTAMINATION = 0.01
FIT_SAMPLE_SIZE = 5_000 if RUN_MODE == "sample" else 200_000
BATCH_SIZE = 2_048 if RUN_MODE == "sample" else 16_384
MIN_EXPECTED_ROWS = 100 if RUN_MODE == "sample" else 1_000_000

if not ACTIVE_PROCESSED_PATH.exists():
    raise FileNotFoundError(f"{ACTIVE_PROCESSED_PATH} not found. Run 02_preprocess.ipynb first.")
row_count = parquet_row_count(ACTIVE_PROCESSED_PATH)
if row_count < MIN_EXPECTED_ROWS:
    raise RuntimeError(f"{ACTIVE_PROCESSED_PATH} has only {row_count:,} rows. Run 02_preprocess.ipynb first.")

embedding_path = ACTIVE_EMBEDDING_DIR / f'{ENCODER_TO_TEST}.npy'
if not embedding_path.exists():
    raise FileNotFoundError(f"{embedding_path} not found. Run 03_encode.ipynb first or change ENCODER_TO_TEST.")

print(f"Run mode: {RUN_MODE}; rows: {row_count:,}; embedding: {embedding_path}")


## 2. Run Anomaly Scoring


In [ ]:
summary = run_full_anomaly_for_embedding(
    processed_path=ACTIVE_PROCESSED_PATH,
    embedding_path=embedding_path,
    results_dir=ACTIVE_EXPERIMENTS_DIR / 'anomaly',
    figures_dir=ACTIVE_EXPERIMENTS_DIR / 'figures',
    contamination=CONTAMINATION,
    fit_sample_size=FIT_SAMPLE_SIZE,
    batch_size=BATCH_SIZE,
)

display(pd.Series(summary).to_frame('value'))


## 3. Inspect Saved CSVs


In [ ]:
iso_path = Path(summary['iso_csv'])
inc_path = Path(summary['inconsistency_csv'])

print(f"Isolation anomaly CSV: {iso_path}")
print(f"Rating inconsistency CSV: {inc_path}")

if iso_path.exists():
    display(pd.read_csv(iso_path).head(5))
if inc_path.exists():
    display(pd.read_csv(inc_path).head(5))
